In [ ]:
import pandas as pd

df = pd.read_csv(
    "arrhythmia.csv",
    header=None,
    na_values="?"
)

print("Raw shape:", df.shape)
print(df.head())

In [ ]:
df.columns = [
    "age", "sex", "height", "weight",
    "qrs_duration", "pr_interval", "qt_interval",
    "t_interval", "p_interval", "qrs_angle",
    "t_angle", "p_angle", "qrst_angle",
    "j_angle", "heart_rate"
] + list(df.columns[15:-1]) + ["class"]

print(df.columns[:15])
print(df.columns[-3:])
print(df.head())

In [ ]:
df_original = df.copy(deep=True)

print("Records:", df_original.shape[0])
print("Columns:", df_original.shape[1])

print("\nData-type counts:")
print(df_original.dtypes.value_counts())

df_original.info()

print("\nSummary of numeric columns:")
print(df_original.describe())

In [ ]:
missing = df_original.isna().sum()

print("Columns containing missing values:")
print(missing[missing > 0].sort_values(ascending=False))

print("\nNumber of duplicate rows:")
print(df_original.duplicated().sum())

range_cols = ["age", "height", "weight", "heart_rate"]

print("\nMinimum and maximum values:")
print(df_original[range_cols].agg(["min", "max"]))

print("\nPatients per diagnosis class:")
print(df_original["class"].value_counts().sort_index())

In [ ]:
review_cols = ["age", "sex", "height", "weight", "class"]

records_to_review = df_original.loc[
    (df_original["age"] < 18)
    | (df_original["height"] < 120)
    | (df_original["height"] > 220)
    | (df_original["weight"] < 25),
    review_cols
].sort_values(["age", "height", "weight"])

records_to_review = records_to_review.copy()

records_to_review["bmi"] = (
    records_to_review["weight"]
    / (records_to_review["height"] / 100) ** 2
)

print(records_to_review.round(2).to_string())

In [ ]:
def clean_data(data):
    clean = data.copy(deep=True)

    clean["sex"] = clean["sex"].replace(
        {"M": 0, "F": 1, "m": 0, "f": 1}
    )

    clean["bmi"] = (
        clean["weight"]
        / (clean["height"] / 100) ** 2
    )

    invalid_record = (
        (clean["height"] > 250)
        | ((clean["age"] <= 1) & (clean["height"] > 100))
        | ((clean["age"] >= 20) & (clean["bmi"] < 10))
    )

    print("Clearly unreliable records removed:")
    print(
        clean.loc[
            invalid_record,
            ["age", "sex", "height", "weight", "bmi", "class"]
        ].round(2)
    )

    clean = clean.loc[~invalid_record].copy()

    clean["bmi"] = (
        clean["weight"]
        / (clean["height"] / 100) ** 2
    )

    for col in ["p_angle", "t_angle", "qrst_angle", "heart_rate"]:
        clean[col] = clean[col].fillna(clean[col].median())

    clean = clean.drop(columns=["j_angle"], errors="ignore")
    clean = clean.drop_duplicates()
    clean["arrhythmia_present"] = clean["class"] != 1
    clean = clean.reset_index(drop=True)

    return clean

In [ ]:
df_etl_clean = clean_data(df_original)

print("Original shape:", df_original.shape)
print("Cleaned shape:", df_etl_clean.shape)
print("Rows removed:", len(df_original) - len(df_etl_clean))

print("\nCleaned demographic ranges:")
print(
    df_etl_clean[
        ["age", "height", "weight", "bmi"]
    ].agg(["min", "max"])
)

print(
    "Heights above 250 cm:",
    (df_etl_clean["height"] > 250).sum()
)

print(
    "Adults with BMI below 10:",
    (
        (df_etl_clean["age"] >= 20)
        & (df_etl_clean["bmi"] < 10)
    ).sum()
)

print("Duplicate rows:", df_etl_clean.duplicated().sum())
print("j_angle still present:", "j_angle" in df_etl_clean.columns)

In [ ]:
import sqlite3

# ETL: transform first, then load
with sqlite3.connect("etl.db") as conn:
    df_etl_clean.to_sql(
        "clean",
        conn,
        if_exists="replace",
        index=False
    )

# ELT: load raw first, then transform
with sqlite3.connect("elt.db") as conn:
    df_original.to_sql(
        "raw",
        conn,
        if_exists="replace",
        index=False
    )

    df_elt_raw = pd.read_sql(
        "SELECT * FROM raw",
        conn
    )

    df_elt_clean = clean_data(df_elt_raw)

    df_elt_clean.to_sql(
        "clean",
        conn,
        if_exists="replace",
        index=False
    )

print("ETL clean shape:", df_etl_clean.shape)
print("ELT raw shape:", df_elt_raw.shape)
print("ELT clean shape:", df_elt_clean.shape)

In [ ]:
with sqlite3.connect("etl.db") as conn:
    print("Tables in etl.db:")
    print(
        pd.read_sql(
            "SELECT name FROM sqlite_master WHERE type='table'",
            conn
        )
    )

with sqlite3.connect("elt.db") as conn:
    print("\nTables in elt.db:")
    print(
        pd.read_sql(
            "SELECT name FROM sqlite_master WHERE type='table'",
            conn
        )
    )

### Series versus DataFrame

In [ ]:
print(df_original["heart_rate"].mean())

print(
    df_original[
        ["heart_rate", "age"]
    ].mean()
)

### Loop versus vectorised calculation

In [ ]:
import time

start = time.time()
slow_result = []

for i in range(len(df_original)):
    slow_result.append(
        df_original["weight"].iloc[i]
        / df_original["height"].iloc[i]
    )

slow_time = time.time() - start

start = time.time()

fast_result = (
    df_original["weight"]
    / df_original["height"]
)

fast_time = time.time() - start

print("Slow result:")
print(slow_result[:5])

print("\nFast result:")
print(fast_result.head().tolist())

print("\nSlow time:", slow_time)
print("Fast time:", fast_time)
print("\nFast method is", slow_time / fast_time, "times faster")

In [ ]:
print(df_original.shape)
print(df_original.dtypes.value_counts())
df_original.info()

In [ ]:
summary_cols = ["age", "height", "weight", "heart_rate"]

print(df_etl_clean[summary_cols].describe())

skew_results = (
    df_etl_clean[summary_cols]
    .skew()
    .round(2)
)

print(skew_results)

print(
    df_etl_clean["arrhythmia_present"]
    .value_counts(normalize=True)
)

print("Minimum age retained:", df_etl_clean["age"].min())
print("Minimum weight retained:", df_etl_clean["weight"].min())

In [ ]:
import matplotlib.pyplot as plt

plt.hist(
    df_etl_clean["heart_rate"].dropna(),
    bins=20
)
plt.xlabel("Heart rate (bpm)")
plt.ylabel("Number of patients")
plt.title("Distribution of heart rate")
plt.show()

plt.hist(
    df_etl_clean["age"].dropna(),
    bins=25
)
plt.xlabel("Age")
plt.ylabel("Number of patients")
plt.title("Distribution of age")
plt.show()

In [ ]:
for col, label, unit in [
    ("height", "Height", "cm"),
    ("weight", "Weight", "kg"),
    ("heart_rate", "Heart rate", "bpm"),
]:
    plt.hist(df_original[col].dropna(), bins=30)
    plt.title(f"{label} before cleaning")
    plt.xlabel(f"{label} ({unit})")
    plt.ylabel("Number of patients")
    plt.show()

In [ ]:
for col in ["age", "height", "weight", "heart_rate"]:
    print(
        col,
        "raw range:",
        (df_original[col].min(), df_original[col].max()),
        "clean range:",
        (df_etl_clean[col].min(), df_etl_clean[col].max()),
    )

In [ ]:
for col, label, unit in [
    ("height", "Height", "cm"),
    ("weight", "Weight", "kg"),
    ("heart_rate", "Heart rate", "bpm"),
]:
    plt.hist(df_etl_clean[col].dropna(), bins=30)
    plt.title(f"{label} after cleaning")
    plt.xlabel(f"{label} ({unit})")
    plt.ylabel("Number of patients")
    plt.show()

In [ ]:
plt.boxplot(
    df_etl_clean["weight"].dropna(),
    vert=False,
    tick_labels=["Weight"]
)
plt.xlabel("Weight (kg)")
plt.show()

In [ ]:
def iqr_flag(series):
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    return (
        (series < q1 - 1.5 * iqr)
        | (series > q3 + 1.5 * iqr)
    )

age_flag = iqr_flag(df_etl_clean["age"])
height_flag = iqr_flag(df_etl_clean["height"])
weight_flag = iqr_flag(df_etl_clean["weight"])

demographic_flags = age_flag | height_flag | weight_flag

print(
    df_etl_clean.loc[
        demographic_flags,
        ["age", "sex", "height", "weight", "class"]
    ]
    .sort_values(["age", "height", "weight"])
    .to_string()
)

In [ ]:
mean_hr = df_etl_clean["heart_rate"].mean()
std_hr = df_etl_clean["heart_rate"].std()

z_hr = (
    df_etl_clean["heart_rate"] - mean_hr
) / std_hr

print("Heart-rate values with |z| > 3:")
print(
    df_etl_clean.loc[
        z_hr.abs() > 3,
        ["age", "sex", "heart_rate", "class"]
    ]
)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4))

axes[0].boxplot(
    df_etl_clean["height"].dropna(),
    tick_labels=["Height"]
)
axes[0].set_title("Height")
axes[0].set_ylabel("cm")

axes[1].boxplot(
    df_etl_clean["heart_rate"].dropna(),
    tick_labels=["Heart rate"]
)
axes[1].set_title("Heart rate")
axes[1].set_ylabel("bpm")

axes[2].boxplot(
    df_etl_clean["weight"].dropna(),
    tick_labels=["Weight"]
)
axes[2].set_title("Weight")
axes[2].set_ylabel("kg")

plt.suptitle("Cleaned data before optional capping")
plt.show()

**Task:** Identify at least two flagged but plausible records and explain why they should be retained.

**Your interpretation:**

In [ ]:
df_sensitivity = df_etl_clean.copy(deep=True)

df_sensitivity["height_capped"] = (
    df_sensitivity["height"].clip(
        lower=df_sensitivity["height"].quantile(0.01)
    )
)

df_sensitivity["heart_rate_capped"] = (
    df_sensitivity["heart_rate"].clip(
        lower=df_sensitivity["heart_rate"].quantile(0.01),
        upper=df_sensitivity["heart_rate"].quantile(0.99)
    )
)

df_sensitivity["weight_capped"] = (
    df_sensitivity["weight"].clip(
        upper=df_sensitivity["weight"].quantile(0.99)
    )
)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4))

axes[0].boxplot(
    df_sensitivity["height_capped"].dropna(),
    tick_labels=["Height"]
)
axes[0].set_title("Height (sensitivity-capped)")
axes[0].set_ylabel("cm")

axes[1].boxplot(
    df_sensitivity["heart_rate_capped"].dropna(),
    tick_labels=["Heart rate"]
)
axes[1].set_title("Heart rate (capped)")
axes[1].set_ylabel("bpm")

axes[2].boxplot(
    df_sensitivity["weight_capped"].dropna(),
    tick_labels=["Weight"]
)
axes[2].set_title("Weight (capped)")
axes[2].set_ylabel("kg")

plt.suptitle("Optional sensitivity-only capped view")
plt.show()

**Task:** State whether capping changes the visual interpretation and why the capped copy is not used as the main dataset.

**Your interpretation:**

In [ ]:
import numpy as np

rate = (
    df_etl_clean
    .groupby("sex", observed=False)["arrhythmia_present"]
    .mean()
)

print(rate)

sex_labels = {0: "Male", 1: "Female"}
labels = [sex_labels[int(value)] for value in rate.index]

plt.bar(labels, rate.values)
plt.xlabel("Sex")
plt.ylabel("Proportion with arrhythmia")
plt.title("Arrhythmia rate by sex")
plt.show()

In [ ]:
num_cols = [
    "age", "height", "weight", "qrs_duration",
    "pr_interval", "qt_interval", "heart_rate"
]

corr_matrix = df_etl_clean[num_cols].corr()
print(corr_matrix.round(2))

upper_triangle = corr_matrix.where(
    np.triu(
        np.ones(corr_matrix.shape),
        k=1
    ).astype(bool)
)

strongest_pair = upper_triangle.abs().stack().idxmax()

print(
    "Strongest absolute correlation:",
    strongest_pair,
    round(
        corr_matrix.loc[
            strongest_pair[0],
            strongest_pair[1]
        ],
        2
    )
)

plt.imshow(
    corr_matrix,
    cmap="RdYlGn",
    vmin=-1,
    vmax=1
)

plt.xticks(
    range(len(num_cols)),
    num_cols,
    rotation=45,
    ha="right"
)

plt.yticks(range(len(num_cols)), num_cols)
plt.colorbar(label="Correlation")
plt.title("Correlation between 7 of 279 input features")
plt.show()

**Task:** Write:
- One paragraph interpreting the sex comparison
- One paragraph interpreting the correlation heatmap

**Your interpretation:**

In [ ]:
class_names = {
    1: "Normal",
    2: "Ischemic changes (CAD)",
    3: "Old Ant. MI",
    4: "Old Inf. MI",
    5: "Sinus tachycardia",
    6: "Sinus bradycardia",
    7: "PVC",
    8: "Supraventricular PC",
    9: "Left bundle branch block",
    10: "Right bundle branch block",
    11: "1st degree AV block",
    12: "2nd degree AV block",
    13: "3rd degree AV block",
    14: "Left vent. hypertrophy",
    15: "Atrial fib./flutter",
    16: "Other",
}

pqrst_cols = [
    "p_interval", "qrs_duration", "pr_interval",
    "qt_interval", "t_interval", "heart_rate"
]

class_counts = (
    df_etl_clean["class"]
    .value_counts()
    .sort_index()
)

print("Patients per class:")
print(class_counts)

profile = (
    df_etl_clean
    .groupby("class")[pqrst_cols]
    .mean()
)

print("Mean profile per class:")
print(profile.round(1))

profile_z = (
    profile - profile.mean()
) / profile.std()

row_labels = [
    class_names[int(c)]
    for c in profile_z.index
]

plt.imshow(
    profile_z.values,
    cmap="RdYlGn",
    vmin=-2,
    vmax=2,
    aspect="auto"
)

plt.xticks(
    range(len(pqrst_cols)),
    pqrst_cols,
    rotation=45,
    ha="right"
)

plt.yticks(range(len(profile_z)), row_labels)
plt.colorbar(label="Standardised mean (per column)")
plt.title("PQRST + heart rate profile, by arrhythmia type")
plt.show()

In [ ]:
miss = (
    df_original
    .isna()
    .mean()
    .sort_values(ascending=False)
)

miss = miss[miss > 0] * 100

plt.barh(
    miss.index.astype(str),
    miss.values
)
plt.xlabel("% missing")
plt.title("Missing values in the raw data")
plt.show()

In [ ]:
order = sorted(df_etl_clean["class"].unique())

x_pos = df_etl_clean["class"].map(
    {c: i for i, c in enumerate(order)}
)

plt.scatter(
    x_pos,
    df_etl_clean["heart_rate"],
    alpha=0.4
)

means = (
    df_etl_clean
    .groupby("class")["heart_rate"]
    .mean()
)

print("Mean heart rate by class:")
print(means.sort_values())

plt.scatter(
    range(len(order)),
    [means[c] for c in order],
    color="red",
    marker="D",
    label="Mean per type"
)

plt.xticks(
    range(len(order)),
    [class_names[c] for c in order],
    rotation=45,
    ha="right"
)

plt.xlabel("Arrhythmia type")
plt.ylabel("Heart rate (bpm)")
plt.title("Heart rate by arrhythmia type")
plt.legend()
plt.show()

In [ ]:
colors = df_etl_clean["arrhythmia_present"].map(
    {True: "orange", False: "teal"}
)

plt.scatter(
    df_etl_clean["height"],
    df_etl_clean["weight"],
    c=colors,
    alpha=0.5
)

plt.xlabel("Height (cm)")
plt.ylabel("Weight (kg)")
plt.title("Height vs weight, by diagnosis")
plt.show()

In [ ]:
jitter = (
    df_etl_clean["arrhythmia_present"].astype(int)
    + np.random.uniform(
        -0.08,
        0.08,
        len(df_etl_clean)
    )
)

plt.scatter(
    df_etl_clean["age"],
    jitter,
    c=colors,
    alpha=0.5
)

plt.yticks([0, 1], ["No", "Yes"])
plt.xlabel("Age")
plt.ylabel("arrhythmia_present")
plt.title("Age vs arrhythmia_present")
plt.show()

In [ ]:
cols = [
    "height", "weight",
    "qt_interval", "heart_rate"
]

fig, axes = plt.subplots(4, 4, figsize=(9, 9))

for i, c1 in enumerate(cols):
    for j, c2 in enumerate(cols):
        ax = axes[i, j]

        if i == j:
            ax.hist(
                df_etl_clean[c1].dropna(),
                bins=20
            )
        else:
            ax.scatter(
                df_etl_clean[c2],
                df_etl_clean[c1],
                s=4,
                alpha=0.35,
                c=colors
            )

        if i == 3:
            ax.set_xlabel(c2)

        if j == 0:
            ax.set_ylabel(c1)

plt.tight_layout()

from matplotlib.patches import Patch

fig.legend(
    handles=[
        Patch(color="teal", label="No arrhythmia"),
        Patch(color="orange", label="Arrhythmia present"),
    ],
    loc="upper right"
)

plt.show()

In [ ]:
print(
    "Sample mean age:",
    df_etl_clean["age"].mean()
)

In [ ]:
from scipy import stats

a = df_etl_clean.loc[
    df_etl_clean["arrhythmia_present"],
    "heart_rate"
]

b = df_etl_clean.loc[
    ~df_etl_clean["arrhythmia_present"],
    "heart_rate"
]

t, p = stats.ttest_ind(
    a,
    b,
    equal_var=False
)

print("Mean with arrhythmia:", a.mean())
print("Mean without arrhythmia:", b.mean())
print(f"t = {t:.2f}, p = {p:.4f}")

if p < 0.05:
    print(
        "Reject H0: the mean heart rates "
        "differ significantly."
    )
else:
    print(
        "Fail to reject H0: no significant "
        "mean difference was detected."
    )

In [ ]:
table = pd.crosstab(
    df_etl_clean["sex"],
    df_etl_clean["arrhythmia_present"]
)

print(table)

chi2, p, dof, expected = stats.chi2_contingency(table)

print(f"chi2 = {chi2:.2f}, p = {p:.6f}")
print("Degrees of freedom:", dof)
print("Expected counts:")
print(expected)

if p < 0.05:
    print(
        "Reject H0: sex and diagnosis are "
        "associated in this sample."
    )
else:
    print(
        "Fail to reject H0: no significant "
        "association was detected."
    )

In [ ]:
print(df_etl_clean.shape)

clinical_cols = ["age", "sex", "height", "weight", "qrs_duration",
"pr_interval", "qt_interval", "t_interval", "p_interval",
"qrs_angle", "t_angle", "p_angle", "qrst_angle", "bmi"]

correlations = df_etl_clean[clinical_cols + ["heart_rate"]].corr()["heart_rate"]
print(correlations.abs().sort_values(ascending=False))

In [50]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
X = df_etl_clean[["qt_interval"]]
y = df_etl_clean["heart_rate"]
print("X:", X.shape, " y:", y.shape)

X: (448, 1)  y: (448,)


In [51]:
X_train, X_test, y_train, y_test = train_test_split(
X, y, test_size=0.3, random_state=42
)
print("Train:", X_train.shape, " Test:", X_test.shape)

Train: (313, 1)  Test: (135, 1)


In [52]:
model = LinearRegression()
model.fit(X_train, y_train)
print("Model trained on", len(X_train), "patients")

Model trained on 313 patients


In [53]:
print("Coefficients:", model.coef_)
print("Intercept:", model.intercept_)

Coefficients: [-0.24785936]
Intercept: 165.1525239758605


In [54]:
predictions = model.predict(X_test)
comparison = pd.DataFrame({"actual": y_test, "predicted": predictions})
print(comparison.head())

     actual  predicted
285    72.0  70.470248
296    73.0  70.470248
117    93.0  77.658169
346    63.0  72.453123
70     69.0  76.914591


In [55]:
r2 = r2_score(y_test, predictions)
print("R²:", round(r2, 2))

R²: 0.34


In [56]:
target = df_etl_clean["arrhythmia_present"].astype(int)
target_corr = df_etl_clean[clinical_cols].corrwith(target).abs()
print(target_corr.sort_values(ascending=False))

qrs_duration    0.333656
sex             0.224763
t_interval      0.220525
p_interval      0.090729
qrs_angle       0.090267
pr_interval     0.047034
qrst_angle      0.037458
age             0.035833
bmi             0.030163
qt_interval     0.026888
weight          0.013643
height          0.010807
t_angle         0.001403
p_angle         0.000545
dtype: float64


In [58]:
feature_corr = df_etl_clean[clinical_cols].corr().abs()
print(feature_corr.round(2))

               age   sex  height  weight  qrs_duration  pr_interval  \
age           1.00  0.07    0.22    0.35          0.01         0.03   
sex           0.07  1.00    0.49    0.28          0.34         0.05   
height        0.22  0.49    1.00    0.58          0.04         0.07   
weight        0.35  0.28    0.58    1.00          0.09         0.12   
qrs_duration  0.01  0.34    0.04    0.09          1.00         0.02   
pr_interval   0.03  0.05    0.07    0.12          0.02         1.00   
qt_interval   0.14  0.07    0.02    0.04          0.22         0.07   
t_interval    0.01  0.19    0.05    0.14          0.40         0.07   
p_interval    0.09  0.09    0.13    0.12          0.05         0.67   
qrs_angle     0.24  0.08    0.11    0.15          0.14         0.01   
t_angle       0.01  0.14    0.04    0.04          0.03         0.12   
p_angle       0.07  0.00    0.04    0.06          0.04         0.05   
qrst_angle    0.27  0.04    0.12    0.19          0.07         0.03   
bmi   

In [60]:
threshold = 0.7
to_drop = set()
for i in range(len(clinical_cols)):
    for j in range(i + 1, len(clinical_cols)):
        col_i, col_j = clinical_cols[i], clinical_cols[j]
        if feature_corr.loc[col_i, col_j] > threshold:
            if target_corr[col_i] >= target_corr[col_j]:
                to_drop.add(col_j)
            else:
                to_drop.add(col_i)
selected_features = [c for c in clinical_cols if c not in to_drop]
print("Dropped as redundant:", sorted(to_drop))
print("Selected features:", selected_features)

Dropped as redundant: ['qrst_angle', 'weight']
Selected features: ['age', 'sex', 'height', 'qrs_duration', 'pr_interval', 'qt_interval', 't_interval', 'p_interval', 'qrs_angle', 't_angle', 'p_angle', 'bmi']


In [62]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix
X = df_etl_clean[selected_features]
y = df_etl_clean["arrhythmia_present"]
print("X:", X.shape)
print(y.value_counts())

X: (448, 12)
arrhythmia_present
False    244
True     204
Name: count, dtype: int64


In [63]:
X_train, X_test, y_train, y_test = train_test_split(
X, y, test_size=0.3, random_state=42, stratify=y
)
print("Train:", X_train.shape, " Test:", X_test.shape)
print("Train arrhythmia rate:", round(y_train.mean(), 2))
print("Test arrhythmia rate:", round(y_test.mean(), 2))

Train: (313, 12)  Test: (135, 12)
Train arrhythmia rate: 0.46
Test arrhythmia rate: 0.45


In [64]:
model = LogisticRegression(max_iter=2000)
model.fit(X_train, y_train)
probabilities = model.predict_proba(X_test)
print(probabilities[:5])

[[0.38231542 0.61768458]
 [0.75527884 0.24472116]
 [0.67145989 0.32854011]
 [0.39628563 0.60371437]
 [0.65055591 0.34944409]]


In [65]:
predictions = model.predict(X_test)
accuracy = accuracy_score(y_test, predictions)
print("Accuracy:", round(accuracy, 2))

Accuracy: 0.68


In [66]:
baseline_guess = y_train.value_counts().idxmax()
baseline_accuracy = (y_test == baseline_guess).mean()
print("Baseline:", round(baseline_accuracy, 2))

Baseline: 0.55


In [67]:
cm = confusion_matrix(y_test, predictions)
print(cm)

[[56 18]
 [25 36]]


In [68]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_etl_clean[selected_features])
print(X_scaled[:3])

[[ 1.75679487 -1.11355287  2.59468087  0.13199286  0.84012028  0.09452325
   0.10758421  1.19690286 -1.09230811 -0.40423213  0.51069912 -0.64888757]
 [ 0.57514035  0.89802651  0.10812095 -0.51710486  0.41730329  1.03001113
  -0.59337169 -1.97663907 -0.18248064  0.01303975 -2.34657032 -0.37756178]
 [ 0.45075567 -1.11355287  0.80435773  3.18275215  0.1725145   0.56226719
   0.41600481  0.46156997  1.39307424 -0.03911924  0.72234871  1.35538812]]


In [69]:
from sklearn.cluster import KMeans
kmeans = KMeans(n_clusters=2, random_state=42, n_init=10)
df_etl_clean["cluster"] = kmeans.fit_predict(X_scaled)
print(df_etl_clean["cluster"].value_counts())

cluster
1    248
0    200
Name: count, dtype: int64


In [70]:
print(pd.crosstab(df_etl_clean["cluster"], df_etl_clean["arrhythmia_present"]))

arrhythmia_present  False  True 
cluster                         
0                      84    116
1                     160     88
